<a href="https://colab.research.google.com/github/valeryviviana/PruebaR/blob/master/Etapa_1_Proyecto_PrincipiosML_Valery_Casta%C3%B1eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solución Etapa 1 del proyecto. La tarea de regresión: modelos polinomiales y regularizados

El problema que se va abordar es: construir un modelo predictivo que permita determinar la demanda sobre el uso de
un sistema de alquiler de bicicletas. Este conocimiento puede dar soporte para mejorar el servicio y conocer los
factores que inciden en su eficiencia.

## Objetivos
- Aplicar técnicas de regresión para construir un modelo predictivo que permita estimar la demanda sobre el uso de un
sistema de alquiler de bicicletas siguiendo el ciclo de machine learning.
- Determinar cuáles son los factores que mas inciden en la demanda con base en los datos.


# 1. Importación de librerías requeridas


In [1]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import f_regression
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler, RobustScaler
from sklearn.pipeline import make_pipeline

from importlib.metadata import version
print(f"Versión de Scikit-learn: {version('scikit-learn')}")
print(f"Versión de Pandas: {version('pandas')}")


Versión de Scikit-learn: 1.6.1
Versión de Pandas: 2.2.2


## 2. Carga de datos


In [2]:
data_raw = pd.read_excel('/content/Maestria/Datos_Etapa-1.xlsx')
data_raw.head()

,season,weekday,weathersit,temp,atemp,hum,windspeed,cnt,time_of_day
0,Winter,6,Clear,3.28,3.0014,0.81,0.0,16,Night
1,Winter,6,Clear,2.34,1.9982,0.80,0.0,40,Night
2,Winter,6,Clear,2.34,1.9982,0.80,0.0,32,Night
3,Winter,6,Clear,3.28,3.0014,0.75,0.0,13,Night
4,Winter,6,Clear,3.28,3.0014,0.75,0.0,1,Night


# 3. Diccionario de los datos

In [3]:
dictionary_data = pd.read_excel('/content/Maestria/DiccionarioDatos_Etapa-1.xlsx')
dictionary_data

,Columna,Tipo,Descripción
0,season,categórica,"Estación del año (Winter, Spring, Summer, Fall)"
1,weekday,numérico,Día de la semana (de 1 a 7)
2,weathersit,categórica,"Clima (Clear, Mist, Light Rain, Heavy Rain)"
3,temp,numérico,Temperatura
4,atemp,numérico,Sensación de temperatura
5,hum,numérico,Humedad
6,windspeed,numérico,Velocidad del viento
7,cnt,numérico,Cantidad de bicicletas rentadas
8,time_of_day,categórica,"Parte del día (Morning, Evening, Night)"


## 4. Análisis Descriptivo de los Datos



In [4]:
print("Información general del DataFrame data_raw:")
data_raw.info()

Información general del DataFrame data_raw:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   season       17379 non-null  object 
 1   weekday      17379 non-null  int64  
 2   weathersit   17379 non-null  object 
 3   temp         17379 non-null  float64
 4   atemp        17379 non-null  float64
 5   hum          17379 non-null  float64
 6   windspeed    17379 non-null  float64
 7   cnt          17379 non-null  int64  
 8   time_of_day  17379 non-null  object 
dtypes: float64(4), int64(2), object(3)
memory usage: 1.2+ MB


In [5]:
print("Estadísticas descriptivas de data_raw:")
display(data_raw.describe(include='all'))

Estadísticas descriptivas de data_raw:


,season,weekday,weathersit,temp,atemp,hum,windspeed,cnt,time_of_day
count,17379,17379.000000,17379,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379
unique,4,NaN,4,NaN,NaN,NaN,NaN,NaN,3
top,Summer,NaN,Clear,NaN,NaN,NaN,NaN,NaN,Night
freq,4496,NaN,11413,NaN,NaN,NaN,NaN,NaN,6471
mean,NaN,3.003683,NaN,15.358397,15.401157,0.627229,12.736540,189.463088,NaN
std,NaN,2.005771,NaN,9.050138,11.342114,0.192930,8.196795,181.387599,NaN
min,NaN,0.000000,NaN,-7.060000,-16.000000,0.000000,0.000000,1.000000,NaN
25%,NaN,1.000000,NaN,7.980000,5.997800,0.480000,7.001500,40.000000,NaN
50%,NaN,3.000000,NaN,15.500000,15.996800,0.630000,12.998000,142.000000,NaN
75%,NaN,5.000000,NaN,23.020000,24.999200,0.780000,16.997900,281.000000,NaN


In [6]:
print("Valores nulos por columna en data_raw:")
display(data_raw.isnull().sum())

Valores nulos por columna en data_raw:


,0
season,0
weekday,0
weathersit,0
temp,0
atemp,0
hum,0
windspeed,0
cnt,0
time_of_day,0


In [7]:
print("Valores únicos en la columna 'weekday':")
display(data_raw['weekday'].value_counts().sort_index())

Valores únicos en la columna 'weekday':


,count
weekday,
0,2502
1,2479
2,2453
3,2475
4,2471
5,2487
6,2512


In [8]:
data = data_raw.copy()

## Eliminación de datos faltantes y duplicados

In [9]:
data.isna()

,season,weekday,weathersit,temp,atemp,hum,windspeed,cnt,time_of_day
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
17374,False,False,False,False,False,False,False,False,False
17375,False,False,False,False,False,False,False,False,False
17376,False,False,False,False,False,False,False,False,False
17377,False,False,False,False,False,False,False,False,False


In [10]:
data.isna().sum()

,0
season,0
weekday,0
weathersit,0
temp,0
atemp,0
hum,0
windspeed,0
cnt,0
time_of_day,0


In [11]:
data.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
17374,False
17375,False
17376,False
17377,False


In [12]:
data.duplicated().sum()

np.int64(42)

In [13]:
data.shape

(17379, 9)

# 5. División de datos
Vamos a separar la variable objetivo y las variables independientes para, posteriormente, crear los conjuntos de entrenamiento y pruebas:


In [14]:
x = data.drop(['cnt'], axis="columns")
y = data['cnt']


In [15]:
# Verificar que el conjunto de datos no tienen la variable objetivo
x.head()

,season,weekday,weathersit,temp,atemp,hum,windspeed,time_of_day
0,Winter,6,Clear,3.28,3.0014,0.81,0.0,Night
1,Winter,6,Clear,2.34,1.9982,0.80,0.0,Night
2,Winter,6,Clear,2.34,1.9982,0.80,0.0,Night
3,Winter,6,Clear,3.28,3.0014,0.75,0.0,Night
4,Winter,6,Clear,3.28,3.0014,0.75,0.0,Night


Definiremos los conjuntos de entrenamiento y pruebas usando `train_test_split()`, utilizando el conjunto de variables independientes (`x`) y la variable objetivo (`y`):

Con base en el enunciado se tiene en cuenta los siguiente:
- Al hacer la división entrenamiento – test utiliza un valor de semilla de 77 (random_state).


In [16]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=77)

# 6. Transformación de los datos

La variable "weekday" se le ajusta el rango con respecto al que esta definido en el diccionario de datos, se asume que fue un error dado que el rango empieza en 0 y termina 6 esto se evidencio en el analisis descriptivo de los datos, pero en el diccionario se especifica que empieza en 1 y termina en 7.
Adicionalmente se le cambia el tipo de numero a categorico.

In [17]:
x_train['weekday'] = x_train['weekday'] + 1
x_train['weekday'] = x_train['weekday'].astype('category')

In [18]:
x_test['weekday'] = x_test['weekday'] + 1
x_test['weekday'] = x_test['weekday'].astype('category')

In [19]:
corr = x_train.join(y_train).corr(numeric_only=True)
corr['cnt'].sort_values()

,cnt
hum,-0.331177
windspeed,0.095056
atemp,0.403618
temp,0.406648
cnt,1.000000


Con en analisis de correlación se identifica que las variables numericas mas representativas para determinar la demanda del uso del sistema de alquiler de bicicletas son "Temperatura" y "Sensación de temperatura" la "Humedad" y "Velocidad del viento" tienen poca relevancia con la variable objetivo y por ello no tiene sentido incluirlas en el modelo predictivo.

Es importante resaltar que en el diccionario de datos el dia de la semana estaba definido como un valor numerico, pero en la practica esto no da mucho sentido puesto que los días de la semana no representan cantidad, no tienen distancia matemática real, el modelo podria interpretar que 7>1, por ello se tomo la decisión de cambiar esta variable a tipo categorico.

## Transformación de variables categoricas

Para ver cuales variables categoricas son mas relevantes para la variable objetivo se transforman por medio de one hot encoding y luego se aplica ANOVA (F-test) para medir la relación líneal significativa entre cada variable y la variable que se va a predecir.

In [20]:
# One-hot encoding SOLO en train
x_train = pd.get_dummies(x_train, drop_first=True)
print(x_train.columns)
f_values, p_values = f_regression(x_train, y_train)

resultados = pd.DataFrame({
    'variable': x_train.columns,
    'p_value': p_values
}).sort_values('p_value')

resultados

Index(['temp', 'atemp', 'hum', 'windspeed', 'season_Spring', 'season_Summer',
       'season_Winter', 'weekday_2', 'weekday_3', 'weekday_4', 'weekday_5',
       'weekday_6', 'weekday_7', 'weathersit_Heavy Rain',
       'weathersit_Light Rain', 'weathersit_Mist', 'time_of_day_Morning',
       'time_of_day_Night'],
      dtype='object')


,variable,p_value
0,temp,0.000000e+00
1,atemp,0.000000e+00
2,hum,0.000000e+00
17,time_of_day_Night,0.000000e+00
6,season_Winter,1.117754e-193
5,season_Summer,1.656552e-74
14,weathersit_Light Rain,3.518249e-53
3,windspeed,2.821173e-29
4,season_Spring,1.232179e-13
15,weathersit_Mist,1.717790e-09


Para interpretar correctamente ANOVA es importante mencionar que:
- p_value >0,05 quiere decir que la variable es relevante
- p_value>0,05 indica que es una variable no significativa y por ello podria ser candidata a eliminarse.

Se puede decir que:
Los días de la semana no estan afectando mucho la variable objetivo, en cambio el clima y la temperatura si son variables que influyen la demanda sobre el alquiler de bicicletas, por ello se decide eliminar la variable de "weekday"

## Identificar colinealidad entre la variable temp y atemp

In [21]:
x_train[['temp','atemp']].corr()

,temp,atemp
temp,1.000000,0.988631
atemp,0.988631,1.000000


Esto evidencia que "temp" y "atemp" estan indicando casi lo mismo, no da sentido mantener las dos variables en el modelo puesto que podria provocar, multicolinealidad o interpretación poco fiable, por ello se toma la decisión de dejar unicamente "temp"

In [22]:
variables = x_train.columns

variables_significativas = variables[
    ~variables.str.startswith('weekday')  # elimina todas las dummies de weekday
]

variables_significativas = variables_significativas[
    variables_significativas != 'atemp'   # elimina atemp
]

variables_significativas

Index(['temp', 'hum', 'windspeed', 'season_Spring', 'season_Summer',
       'season_Winter', 'weathersit_Heavy Rain', 'weathersit_Light Rain',
       'weathersit_Mist', 'time_of_day_Morning', 'time_of_day_Night'],
      dtype='object')

# 7. Filtro de variables en el conjunto de entrenamiento


In [23]:
x_train = x_train[variables_significativas]

In [24]:
x_train.head()

,temp,hum,windspeed,season_Spring,season_Summer,season_Winter,weathersit_Heavy Rain,weathersit_Light Rain,weathersit_Mist,time_of_day_Morning,time_of_day_Night
326,-0.48,0.59,6.0032,False,False,True,False,False,True,False,True
694,-0.48,0.93,0.0000,False,False,True,False,True,False,True,False
16787,5.16,0.75,11.0014,False,False,False,False,False,True,True,False
13096,31.48,0.53,8.9981,False,True,False,False,False,False,False,True
17193,0.46,0.80,8.9981,False,False,True,False,False,False,True,False


# Modelo regresión polinomial

In [25]:
polynomial_regression = make_pipeline(
    PolynomialFeatures(),
    RobustScaler(),
    LinearRegression()
)


In [26]:
param_grid = {'polynomialfeatures__degree': [2, 3]}

In [27]:
kfold = KFold(n_splits=10, shuffle=True, random_state=0)

In [28]:
modelos_grid = GridSearchCV(polynomial_regression, param_grid, cv=kfold, n_jobs=-1)

In [29]:
%%time
modelos_grid.fit(x_train, y_train)

CPU times: user 1.24 s, sys: 71.7 ms, total: 1.31 s
Wall time: 23.1 s


GridSearchCV(cv=KFold(n_splits=10, random_state=0, shuffle=True),
             estimator=Pipeline(steps=[('polynomialfeatures',
                                        PolynomialFeatures()),
                                       ('robustscaler', RobustScaler()),
                                       ('linearregression',
                                        LinearRegression())]),
             n_jobs=-1, param_grid={'polynomialfeatures__degree': [2, 3]})

In [30]:
print("Mejor parámetro: ", modelos_grid.best_params_)

Mejor parámetro:  {'polynomialfeatures__degree': 3}


In [31]:
mejor_modelo = modelos_grid.best_estimator_
mejor_modelo

Pipeline(steps=[('polynomialfeatures', PolynomialFeatures(degree=3)),
                ('robustscaler', RobustScaler()),
                ('linearregression', LinearRegression())])

## Transformación dataset prueba


In [32]:
x_test = pd.get_dummies(x_test, drop_first=True)

 Ajuste de columnas dado que "weathersit_Heavy Rain" existia en el conjunto de entrenamiento pero en el de prueba no, por ello se alinea las columnas del dataset de prueba con el de entrenamiento

In [33]:
x_train, x_test = x_train.align(
    x_test,
    join='left',
    axis=1,
    fill_value=0
)

In [34]:
x_test.columns

Index(['temp', 'hum', 'windspeed', 'season_Spring', 'season_Summer',
       'season_Winter', 'weathersit_Heavy Rain', 'weathersit_Light Rain',
       'weathersit_Mist', 'time_of_day_Morning', 'time_of_day_Night'],
      dtype='object')

## Evaluación del modelo

In [35]:
y_pred = mejor_modelo.predict(x_test)

print(f'------ Mejor modelo de regresión polinomial ----')
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f'R²: {r2_score(y_test, y_pred):.2f}')

------ Mejor modelo de regresión polinomial ----
RMSE: 130.91
MAE: 96.27
R²: 0.47


# Modelo regresión regularizada lasso

In [36]:
lasso = Lasso(max_iter=500)

In [37]:
param_grid = {'alpha': [1, 2, 3, 4, 5]}

In [38]:
kfold = KFold(n_splits=5, shuffle=True, random_state = 0)

In [39]:
modelos_grid_lasso = GridSearchCV(lasso, param_grid, cv=kfold, n_jobs=-1,verbose=2)

In [40]:
%%time
modelos_grid_lasso.fit(x_train, y_train)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
CPU times: user 127 ms, sys: 12.9 ms, total: 140 ms
Wall time: 362 ms


GridSearchCV(cv=KFold(n_splits=5, random_state=0, shuffle=True),
             estimator=Lasso(max_iter=500), n_jobs=-1,
             param_grid={'alpha': [1, 2, 3, 4, 5]}, verbose=2)

In [41]:
print("Mejor parámetro: ", modelos_grid_lasso.best_params_)

Mejor parámetro:  {'alpha': 1}


In [42]:
mejor_modelo_lasso = modelos_grid_lasso.best_estimator_
pd.DataFrame(zip(x_train.columns, mejor_modelo_lasso.coef_),columns=["Variable","Coeficiente"])

,Variable,Coeficiente
0,temp,6.722142
1,hum,-132.756586
2,windspeed,-0.186873
3,season_Spring,-16.015790
4,season_Summer,-40.007589
5,season_Winter,-44.005344
6,weathersit_Heavy Rain,-0.000000
7,weathersit_Light Rain,-32.880045
8,weathersit_Mist,-0.962056
9,time_of_day_Morning,-79.179224


In [43]:
y_pred_lasso = mejor_modelo_lasso.predict(x_test)

print(f'------ Modelo de regresión Lasso ----')
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_lasso):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lasso):.2f}")
print(f'R²: {r2_score(y_test, y_pred_lasso):.2f}')


------ Modelo de regresión Lasso ----
RMSE: 136.82
MAE: 101.87
R²: 0.42


# Comparativo
Elaboración de una tabla comparativa mostrando el rendimiento sobre test de los dos modelos seleccionados (con mejores rendimientos) de las actividades 3 y 4, con las métricas R2, RMSE y MAE.

In [44]:
# Polinomial
r2_poly = r2_score(y_test, y_pred)
rmse_poly = root_mean_squared_error(y_test, y_pred)

# Lasso
r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = root_mean_squared_error(y_test, y_pred_lasso)

print("Polinomial → R2:", r2_poly, " RMSE:", rmse_poly)
print("Lasso      → R2:", r2_lasso, " RMSE:", rmse_lasso)
comparacion = pd.DataFrame({
    "Modelo": ["Polinomial", "Lasso"],
    "R2": [r2_poly, r2_lasso],
    "RMSE": [rmse_poly, rmse_lasso]
})

comparacion

Polinomial → R2: 0.47126142988641484  RMSE: 130.90986540444382
Lasso      → R2: 0.4224653588633892  RMSE: 136.81726344534457


,Modelo,R2,RMSE
0,Polinomial,0.471261,130.909865
1,Lasso,0.422465,136.817263


El modelo polinomial obtuvo un mejor desempeño que Lasso en el conjunto de prueba, presentando mayor R² y menor RMSE. Esto indica que la inclusión de términos no lineales mejora la capacidad predictiva del modelo.

# Con base en el modelo Lasso determinar las variables más importantes para la predicción.

In [45]:
mejor_modelo_lasso = modelos_grid_lasso.best_estimator_
pd.DataFrame(zip(x_train.columns, mejor_modelo_lasso.coef_),columns=["Variable","Coeficiente"])

,Variable,Coeficiente
0,temp,6.722142
1,hum,-132.756586
2,windspeed,-0.186873
3,season_Spring,-16.015790
4,season_Summer,-40.007589
5,season_Winter,-44.005344
6,weathersit_Heavy Rain,-0.000000
7,weathersit_Light Rain,-32.880045
8,weathersit_Mist,-0.962056
9,time_of_day_Morning,-79.179224


El modelo Lasso identificó como variables más relevantes para la predicción: time_of_day_Night, hum y time_of_day_Morning, ya que presentan los coeficientes de mayor magnitud absoluta. La variable weathersit_Heavy Rain fue descartada automáticamente por el modelo al asignársele un coeficiente igual a cero.

# Análisis de resultados.

- ¿Cuál es el grado de la transformación polinomial que fue seleccionado utilizando la técnica de validación?


In [46]:
print("Mejor parámetro: ", modelos_grid.best_params_)

Mejor parámetro:  {'polynomialfeatures__degree': 3}


- ¿Cuál fue el valor de α que fue seleccionado utilizando la técnica de validación para la regresión Lasso?


In [47]:
print("Mejor parámetro: ", modelos_grid_lasso.best_params_)

Mejor parámetro:  {'alpha': 1}


- A partir de la tabla comparativa, ¿Cuál modelo ofrece el mejor rendimiento sobre el conjunto test? ¿Qué interpretación puedes darles a los valores obtenidos sobre las métricas de rendimiento?


El modelo polinomial obtuvo un mejor desempeño que Lasso en el conjunto de prueba, presentando mayor R² y menor RMSE. Esto indica que la inclusión de términos no lineales mejora la capacidad predictiva del modelo.

- ¿Cuáles variables fueron seleccionadas con el modelo Lasso? A partir de estas, ¿Qué interpretación de cara al problema puedes dar? Reflexiona sobre cómo este nuevo conocimiento podría ayudar a tomar decisiones en el contexto del problema.

El modelo Lasso identificó como variables más relevantes para la predicción: time_of_day_Night, hum y time_of_day_Morning, ya que presentan los coeficientes de mayor magnitud absoluta. La variable weathersit_Heavy Rain fue descartada automáticamente por el modelo al asignársele un coeficiente igual a cero.

## Etapa 1 Proyecto
Principios del machine learning

---
*Presentado por: Valery Viviana Castañeda Agudelo*

*Fecha: Febrero 2026*  

*Universidad de los Andes*  
